# 05 · Differentiable flood twin — let gradients decide where to dig

Three parts:
1. **A 2-D flood simulator written in PyTorch** (the inertial shallow-water scheme of
   Bates et al. 2010, the same physics family as LISFLOOD-FP). Because it's torch ops end to
   end, the simulation is *differentiable*: we can take gradients of "flooded volume" with
   respect to "how deep we dig at each candidate site".
2. **A U-Net emulator** trained on simulated storms — answers any what-if in milliseconds.
3. **Gradient-based intervention design**: given an excavation budget, Adam literally descends
   to the allocation of storage capacity across sites that minimises flooding of built-up areas.

**Runtime:** switch Colab to GPU (Runtime → Change runtime type → T4).
**Requires:** notebook 01 outputs (`dem.tif`, `worldcover.tif`, `sinks.csv`).

In [ ]:
import sys, subprocess, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("rasterio")

WORK = "/content/drive/MyDrive/floodtwin" if os.path.isdir("/content/drive/MyDrive") else "/content/floodtwin"
assert os.path.exists(f"{WORK}/dem.tif"), "Run notebook 01 first."

import numpy as np, rasterio, torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Build the model domain
128 x 128 cells at 60 m (a ~7.7 km square over central Patna): big enough to be real, small
enough that autograd through hundreds of timesteps fits in GPU memory.

In [ ]:
CENTER = (25.605, 85.140)        # lat, lon of crop centre — edit to move the window
N30, N = 256, 128                # 256 px at 30 m -> pooled to 128 px at 60 m
DX = 60.0

with rasterio.open(f"{WORK}/dem.tif") as src:
    dem30 = src.read(1).astype("float64"); T = src.transform
with rasterio.open(f"{WORK}/worldcover.tif") as src:
    wc30 = src.read(1)

col0 = int((CENTER[1] - T.c) / T.a) - N30 // 2
row0 = int((CENTER[0] - T.f) / T.e) - N30 // 2
row0 = max(0, min(row0, dem30.shape[0] - N30)); col0 = max(0, min(col0, dem30.shape[1] - N30))
dem_c = dem30[row0:row0 + N30, col0:col0 + N30]
wc_c = wc30[row0:row0 + N30, col0:col0 + N30]

z_np = dem_c.reshape(N, 2, N, 2).mean(axis=(1, 3))            # 60 m bed elevation
wc_np = wc_c[::2, ::2]                                          # 60 m land cover (nearest)

N_TABLE = {10: 0.060, 20: 0.050, 30: 0.040, 40: 0.040, 50: 0.015, 60: 0.030, 80: 0.030, 90: 0.045}
F_TABLE = {10: 10.0, 20: 8.0, 30: 8.0, 40: 8.0, 50: 1.0, 60: 5.0, 80: 0.0, 90: 0.0}   # mm/hr
mann_np = np.vectorize(lambda v: N_TABLE.get(int(v), 0.035))(wc_np)
infil_np = np.vectorize(lambda v: F_TABLE.get(int(v), 5.0))(wc_np) / 1000.0 / 3600.0   # m/s
built_np = (wc_np == 50).astype("float32")

z0 = torch.tensor(z_np, dtype=torch.float32, device=device)
mann = torch.tensor(mann_np, dtype=torch.float32, device=device)
infil = torch.tensor(infil_np, dtype=torch.float32, device=device)
built = torch.tensor(built_np, dtype=torch.float32, device=device)
print("domain:", z0.shape, "| elev range:", float(z0.min()), "-", float(z0.max()), "m | built fraction:",
      round(float(built.mean()), 2))

## The differentiable simulator
State: water depth `h` plus face discharges `qx, qy` (m²/s). Each step: update face flows from
the water-surface slope with an implicit Manning friction term, limit for stability, then update
depths from flux divergence + rain − infiltration. Closed boundaries (conservative for flooding).

In [ ]:
G, HMIN = 9.81, 1e-3

def step(h, qx, qy, z, rain_ms, dt):
    eta = z + h
    # --- x faces (between columns) ---
    hf = torch.clamp(torch.maximum(eta[:, :-1], eta[:, 1:]) - torch.maximum(z[:, :-1], z[:, 1:]), min=0.0)
    Sx = (eta[:, 1:] - eta[:, :-1]) / DX
    nx = 0.5 * (mann[:, :-1] + mann[:, 1:])
    hf73 = torch.clamp(hf, min=HMIN) ** (7.0 / 3.0)
    qx = (qx - G * hf * dt * Sx) / (1.0 + G * dt * nx ** 2 * torch.abs(qx) / hf73)
    qcap = 0.25 * hf * DX / dt
    qx = torch.where(hf > HMIN, torch.clamp(qx, -qcap, qcap), torch.zeros_like(qx))
    # --- y faces (between rows) ---
    hfy = torch.clamp(torch.maximum(eta[:-1, :], eta[1:, :]) - torch.maximum(z[:-1, :], z[1:, :]), min=0.0)
    Sy = (eta[1:, :] - eta[:-1, :]) / DX
    ny = 0.5 * (mann[:-1, :] + mann[1:, :])
    hfy73 = torch.clamp(hfy, min=HMIN) ** (7.0 / 3.0)
    qy = (qy - G * hfy * dt * Sy) / (1.0 + G * dt * ny ** 2 * torch.abs(qy) / hfy73)
    qcapy = 0.25 * hfy * DX / dt
    qy = torch.where(hfy > HMIN, torch.clamp(qy, -qcapy, qcapy), torch.zeros_like(qy))
    # --- continuity ---
    Qx = torch.nn.functional.pad(qx, (1, 1, 0, 0))     # closed boundaries
    Qy = torch.nn.functional.pad(qy, (0, 0, 1, 1))
    dh = dt / DX * (Qx[:, :-1] - Qx[:, 1:] + Qy[:-1, :] - Qy[1:, :])
    h = torch.clamp(h + dh + rain_ms * dt - infil * dt, min=0.0)
    return h, qx, qy

def simulate(z, rain_mm, storm_hr=1.5, total_hr=3.0, dt=10.0, chunk=60, grad=False):
    """Run a storm; return max water depth. rain_mm falls uniformly over storm_hr."""
    h = torch.zeros_like(z)
    qx = torch.zeros(z.shape[0], z.shape[1] - 1, device=z.device)
    qy = torch.zeros(z.shape[0] - 1, z.shape[1], device=z.device)
    hmax = torch.zeros_like(z)
    nsteps = int(total_hr * 3600 / dt)
    rain_steps = int(storm_hr * 3600 / dt)
    rain_rate = rain_mm / 1000.0 / (storm_hr * 3600.0)

    def run_chunk(h, qx, qy, hmax, k0, k1):
        for k in range(k0, k1):
            r = rain_rate if k < rain_steps else 0.0
            h, qx, qy = step(h, qx, qy, z, r, dt)
            hmax = torch.maximum(hmax, h)
        return h, qx, qy, hmax

    k = 0
    while k < nsteps:
        k1 = min(k + chunk, nsteps)
        if grad:
            h, qx, qy, hmax = torch.utils.checkpoint.checkpoint(
                run_chunk, h, qx, qy, hmax, k, k1, use_reentrant=False)
        else:
            h, qx, qy, hmax = run_chunk(h, qx, qy, hmax, k, k1)
        k = k1
    return hmax

# Sanity run: an 80 mm / 1.5 h cloudburst with no interventions.
with torch.no_grad():
    hmax0 = simulate(z0, rain_mm=80.0)
flood_vol0 = float((torch.relu(hmax0 - 0.15) * built).sum() * DX * DX)
print(f"baseline flooded volume on built-up land: {flood_vol0:,.0f} m³")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(z_np, cmap="terrain"); ax[0].set_title("bed elevation (m)")
im = ax[1].imshow(hmax0.cpu(), cmap="Blues", vmax=1.0)
ax[1].set_title("max depth, 80 mm storm (m)"); plt.colorbar(im, ax=ax[1], shrink=0.8)
plt.tight_layout(); plt.savefig(f"{WORK}/twin_sanity.png", dpi=120); plt.show()
# Cross-check: deep blue blobs here should coincide with notebook 01's sinks and
# notebook 03's observed-water frequency map. If they don't, distrust all three less equally —
# the SAR map is the observation.

## Candidate intervention sites
Use notebook 01's ranked sinks that fall inside this window; pad with local minima if needed.
An "intervention" digs the bed down at a site (recharge pit / retention basin), creating storage.

In [ ]:
import pandas as pd

K, RADIUS = 8, 2
sites = []
try:
    sk = pd.read_csv(f"{WORK}/sinks.csv")
    for _, s in sk.iterrows():
        rr, cc = (int(s.row) - row0) // 2, (int(s.col) - col0) // 2
        if 5 <= rr < N - 5 and 5 <= cc < N - 5:
            sites.append((rr, cc))
        if len(sites) == K:
            break
except Exception as e:
    print("sinks.csv unavailable:", e)

hm = hmax0.cpu().numpy().copy()
while len(sites) < K:                                 # pad with deepest remaining cells
    rr, cc = np.unravel_index(np.argmax(hm), hm.shape)
    hm[max(0, rr - 8):rr + 8, max(0, cc - 8):cc + 8] = 0
    if 5 <= rr < N - 5 and 5 <= cc < N - 5:
        sites.append((int(rr), int(cc)))

masks = torch.zeros(K, N, N, device=device)
yy, xx = torch.meshgrid(torch.arange(N, device=device), torch.arange(N, device=device), indexing="ij")
for i, (rr, cc) in enumerate(sites):
    masks[i] = ((yy - rr) ** 2 + (xx - cc) ** 2 <= RADIUS ** 2).float()
site_area = masks.sum(dim=(1, 2)) * DX * DX
site_any = masks.sum(0).clamp(max=1.0)
eval_mask = built * (1.0 - site_any)      # judge flooding on built land EXCLUDING the pits:
print("sites (row,col):", sites)          # deep water inside a pit is the goal, not a flood

def dig_map(theta):
    depths = 3.0 * torch.sigmoid(theta)               # max 3 m excavation per site
    return (depths.view(K, 1, 1) * masks).sum(0), depths

## Train the U-Net emulator (what-ifs in milliseconds)
Dataset: random storms x random dig allocations → max-depth grids from the simulator.
~10–20 min on a T4 for 220 simulations; lower `N_SAMPLES` for a quick pass.

In [ ]:
N_SAMPLES = 220
X, Y = [], []
zmean, zstd = z0.mean(), z0.std()
with torch.no_grad():
    for i in range(N_SAMPLES):
        rain = float(np.random.uniform(20, 150))
        theta = torch.randn(K, device=device) * 2.0
        D, _ = dig_map(theta)
        if np.random.rand() < 0.2:
            D = torch.zeros_like(D)                   # include do-nothing scenarios
        hmax = simulate(z0 - D, rain_mm=rain)
        x = torch.stack([(z0 - D - zmean) / zstd,
                         torch.full_like(z0, rain / 100.0), D / 3.0])
        X.append(x.cpu()); Y.append(hmax.unsqueeze(0).cpu())
        if (i + 1) % 20 == 0:
            print(f"  simulated {i + 1}/{N_SAMPLES}")
X = torch.stack(X); Y = torch.stack(Y)
torch.save({"X": X, "Y": Y}, f"{WORK}/twin_dataset.pt")
print("dataset:", X.shape, Y.shape)

In [ ]:
import torch.nn as nn

class UNet(nn.Module):
    def __init__(s, ch=3):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(),
                                            nn.Conv2d(o, o, 3, padding=1), nn.ReLU())
        s.e1, s.e2, s.e3 = blk(ch, 16), blk(16, 32), blk(32, 64)
        s.pool = nn.MaxPool2d(2)
        s.up2 = nn.ConvTranspose2d(64, 32, 2, 2); s.d2 = blk(64, 32)
        s.up1 = nn.ConvTranspose2d(32, 16, 2, 2); s.d1 = blk(32, 16)
        s.out = nn.Conv2d(16, 1, 1)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(s.pool(e1)); e3 = s.e3(s.pool(e2))
        d2 = s.d2(torch.cat([s.up2(e3), e2], 1))
        d1 = s.d1(torch.cat([s.up1(d2), e1], 1))
        return torch.nn.functional.softplus(s.out(d1))

emu = UNet().to(device)
opt = torch.optim.Adam(emu.parameters(), lr=1e-3)
ntr = int(0.9 * len(X))
Xtr, Ytr, Xv, Yv = X[:ntr].to(device), Y[:ntr].to(device), X[ntr:].to(device), Y[ntr:].to(device)
for ep in range(40):
    perm = torch.randperm(ntr)
    for b in range(0, ntr, 8):
        idx = perm[b:b + 8]
        loss = ((emu(Xtr[idx]) - Ytr[idx]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        vl = ((emu(Xv) - Yv) ** 2).mean().sqrt()
    if ep % 5 == 0:
        print(f"epoch {ep}: val RMSE {float(vl)*100:.1f} cm")
torch.save(emu.state_dict(), f"{WORK}/emulator.pt")

## Gradient-based intervention design
Decision variable: dig depth at each of the K sites. Objective: flooded volume on built-up
land for the design storm, plus a penalty if excavation cost exceeds the budget. We descend
through the **physics itself** (checkpointed autograd through every timestep).

In [ ]:
DESIGN_RAIN = 100.0                                   # mm — your design storm
BUDGET_M3 = 150_000.0                                 # total excavation budget
LAM = 5.0

theta = torch.full((K,), -2.0, device=device, requires_grad=True)
optt = torch.optim.Adam([theta], lr=0.15)
hist = []
for it in range(50):
    D, depths = dig_map(theta)
    hmax = simulate(z0 - D, rain_mm=DESIGN_RAIN, grad=True)
    flood = (torch.relu(hmax - 0.15) * eval_mask).sum() * DX * DX
    cost = (depths * site_area).sum()
    loss = flood + LAM * torch.relu(cost - BUDGET_M3)
    optt.zero_grad(); loss.backward(); optt.step()
    hist.append((float(flood), float(cost)))
    if it % 5 == 0:
        print(f"iter {it:3d}  flooded {float(flood):>12,.0f} m³   excavation {float(cost):>10,.0f} m³")

In [ ]:
with torch.no_grad():
    D_opt, depths_opt = dig_map(theta)
    hmax_base = simulate(z0, rain_mm=DESIGN_RAIN)
    hmax_opt = simulate(z0 - D_opt, rain_mm=DESIGN_RAIN)
    # naive baseline: same budget split equally across sites
    eq_depth = BUDGET_M3 / float(site_area.sum())
    hmax_eq = simulate(z0 - torch.clamp(torch.full((K,), eq_depth, device=device),
                                        max=3.0).view(K, 1, 1).mul(masks).sum(0),
                       rain_mm=DESIGN_RAIN)

def fv(h):
    return float((torch.relu(h - 0.15) * eval_mask).sum() * DX * DX)

print(f"design storm {DESIGN_RAIN:.0f} mm — flooded volume on built-up land:")
print(f"  do nothing      : {fv(hmax_base):>12,.0f} m³")
print(f"  equal split     : {fv(hmax_eq):>12,.0f} m³")
print(f"  gradient-optimal: {fv(hmax_opt):>12,.0f} m³"
      f"   ({100 * (1 - fv(hmax_opt) / max(fv(hmax_base), 1)):.0f}% reduction)")

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].bar(range(K), depths_opt.cpu()); ax[0].set_title("optimal dig depth per site (m)")
ax[0].set_xlabel("candidate site")
im1 = ax[1].imshow(hmax_base.cpu(), cmap="Blues", vmax=1.0); ax[1].set_title("max depth — do nothing")
diff = (hmax_base - hmax_opt).cpu()
im2 = ax[2].imshow(diff, cmap="RdYlGn", vmin=-0.3, vmax=0.3)
ax[2].set_title("depth reduction with optimal dig (green = drier)")
for a, im in [(ax[1], im1), (ax[2], im2)]:
    plt.colorbar(im, ax=a, shrink=0.8)
plt.tight_layout(); plt.savefig(f"{WORK}/optimization_result.png", dpi=120); plt.show()

## What you just built, honestly
- A *physics-grounded, differentiable* what-if machine: change the rain, the budget, the
  sites, the max dig depth — gradients re-plan in minutes; the emulator answers in milliseconds.
- **Limits to state out loud:** 60 m cells blur streets; sewers/pumps are absent; rain is
  spatially uniform; the scheme's friction/infiltration parameters are uncalibrated guesses.
- **Calibration path (this is the research contribution):** treat Manning n and infiltration
  as *learnable tensors*, simulate historical storms from notebook 04's rainfall, and minimise
  disagreement with notebook 03's SAR water masks — by gradient descent, through the same
  simulator. A satellite-calibrated differentiable flood twin of an Indian city would be
  genuinely novel, publishable work.